# CircuitVision · Dataset Expansion Audit (Computer Engineering Laboratory)

**This notebook prepares and audits candidate datasets. It does NOT train a model or overwrite Round 2.**

It collects sources for two independent tasks:
- **Physical component detection:** resistors, breadboards, modules, boards, sensors and lab instruments.
- **Resistor value reading:** individual resistor color bands; requires a separate specialized model.

Choose a Colab Python runtime (GPU not necessary for dataset audit), then **Runtime → Run all**. Your Roboflow API key is entered privately with getpass and is not saved in this notebook.

The final downloadable file is **CircuitVision_Dataset_Audit_Report.zip**, a small report of class labels, annotation formats, box counts, and downloadable source metadata. Full source datasets remain in the temporary Colab workspace. Public repository links and licenses are documented in docs/COMPUTER_ENGINEERING_DATASET_CATALOG.md.

## 1 · Install the metadata and Roboflow download tools

In [ ]:
%pip -q install "roboflow>=1.1" "PyYAML>=6" "pandas>=2.0"
from getpass import getpass
from pathlib import Path
from collections import Counter
import os, json, re, csv, hashlib, zipfile, traceback
import yaml
import pandas as pd
from roboflow import Roboflow
ROOT=Path('/content/circuitvision_dataset_audit')
ROOT.mkdir(exist_ok=True)
print('Ready to audit labeled datasets. GPU is not needed yet.')

## 2 · Select the source datasets

The first two are your original Round 2 sources. Optional sources expand physical lab components and instruments; the color-band sets are analyzed **separately** and must never be merged into the 57-class physical detector. The candidate version numbers are project/version identifiers, not promises that access is granted. If a source cannot export, the audit records its failure and continues.

In [ ]:
DATASETS=[
  dict(name='eesob_original',workspace='android-yolo',project='eesob',version=4,task='physical',required=True,license='Check Roboflow export'),
  dict(name='electronics_original',workspace='vishnupriyas-workspace',project='electronics-components-sveck',version=7,task='physical',required=True,license='Check Roboflow export'),
  dict(name='electronic_component_detection',workspace='personal-8hq68',project='electronic-component-detection-m6sop',version=3,task='physical',required=False,license='CC BY 4.0'),
  dict(name='lab_instruments',workspace='myspace-natcf',project='deteksi-alat-elektronik',version=2,task='physical',required=False,license='CC BY 4.0'),
  dict(name='resistor_band_colors',workspace='image-gpmgb',project='resistor-color-band-detection-fnbak',version=2,task='bands',required=False,license='Public Domain'),
  dict(name='resistor_band_colors_RIT',workspace='rochester-institute-of-technology-dubai-6oda9',project='resistor-color-band-detection5',version=1,task='bands',required=False,license='Public Domain'),
]
print(pd.DataFrame(DATASETS)[['name','task','workspace','project','version','license']])
print('Adjust the list before downloading if any provider requires a different version.')

## 3 · Download accessible exports

The first try uses the same YOLO11 format as Round 2, then YOLOv8 format for compatible object-detection labels. Never paste your API key into a saved cell, GitHub issue, or ChatGPT message.

In [ ]:
api_key=getpass('Roboflow API key (hidden): ').strip()
assert api_key, 'A Roboflow key is needed to download published dataset exports'
rf=Roboflow(api_key=api_key)
downloads={}
failures=[]
for entry in DATASETS:
    dest=ROOT / entry['name']
    found=next(iter(dest.rglob('data.yaml')),None) if dest.exists() else None
    if found:
        downloads[entry['name']]=found.parent
        print('Existing',entry['name'],found.parent)
        continue
    last_error=None
    for fmt in ('yolov11','yolov8'):
        try:
            exported=rf.workspace(entry['workspace']).project(entry['project']).version(entry['version']).download(fmt,location=str(dest))
            downloads[entry['name']]=Path(exported.location)
            print('Downloaded',entry['name'],'as',fmt)
            last_error=None
            break
        except Exception as e:
            last_error=str(e)
    if last_error:
        failures.append({'name':entry['name'],'error':last_error[:500],'required':entry['required']})
        print('Unavailable',entry['name'],last_error[:180])
del api_key,rf
print('Successful:',len(downloads),'Unavailable:',len(failures))
if any(item['required'] for item in failures):
    print('WARNING: One of the original Round 2 datasets failed to download. Resolve its access before attempting any merged retraining.')

## 4 · Audit class mappings, bounding-box formats, and count labels

Reports the exact source label spelling. Synonyms and split leakage require human review; this notebook **does not blindly combine classes**.

In [ ]:
def read_names(yaml_file):
    data=yaml.safe_load(yaml_file.read_text())
    names=data.get('names',[])
    if isinstance(names,dict):names=[names[k] for k in sorted(names,key=lambda x:int(x))]
    return [str(n) for n in names]
def audit_source(entry,path):
    yaml_file=next(iter(Path(path).rglob('data.yaml')),None)
    if yaml_file is None:return {'name':entry['name'],'problem':'data.yaml missing'}
    names=read_names(yaml_file)
    root=yaml_file.parent
    formats=Counter()
    count=Counter()
    images=Counter()
    bad=[]
    for split in ('train','valid','val','test'):
        labels=root/split/'labels'
        imgs=root/split/'images'
        if imgs.exists():images[split]=len([x for x in imgs.iterdir() if x.suffix.lower() in ('.jpg','.jpeg','.png','.webp')])
        if not labels.exists():continue
        for txt in labels.glob('*.txt'):
            for line in txt.read_text().splitlines():
                parts=line.split()
                if not parts:continue
                try:
                    cls=int(float(parts[0]))
                    if not (0<=cls<len(names)):raise ValueError('class out of range')
                    nums=[float(v) for v in parts[1:]]
                    if not (len(nums)==4 or (len(nums)>=6 and len(nums)%2==0)):raise ValueError('unsupported annotation length')
                    if not all(0<=v<=1 for v in nums):raise ValueError('coordinates out of bounds')
                    fmt='bbox' if len(nums)==4 else 'polygon'
                    formats[fmt]+=1
                    count[(split,names[cls])]+=1
                except Exception as exc:
                    if len(bad)<15:bad.append(str(txt.name)+': '+str(exc))
    return dict(name=entry['name'],task=entry['task'],source_url='https://universe.roboflow.com/'+entry['workspace']+'/'+entry['project'],
        version=entry['version'],license_as_listed=entry['license'],class_count=len(names),classes=names,
        image_counts=dict(images),annotation_formats=dict(formats),
        label_counts=[{'split':sp,'class':c,'boxes':n} for (sp,c),n in sorted(count.items())],sample_errors=bad)
audits=[audit_source(spec,downloads[spec['name']]) for spec in DATASETS if spec['name'] in downloads]
print(pd.DataFrame([{'name':x['name'],'task':x.get('task'),'classes':x.get('class_count'),
 'images':sum(x.get('image_counts',{}).values()),'boxes':sum(x.get('annotation_formats',{}).values()),
 'errors':len(x.get('sample_errors',[]))} for x in audits]).to_string(index=False))
for x in audits:
    print('\n',x['name'],'(',x['task'],'):',', '.join(x.get('classes',[])[:80]))
print('Potential naming collisions must be manually reviewed before any Round 3 training.')

## 5 · Export a compact audit report (not a trained model)

**Send this ZIP to ChatGPT for source-by-source class mapping, synonym review, count checks and preparation for Round 3.** Source images are not included, and nothing is uploaded publicly.

In [ ]:
report={'audit_version':1,'project':'CircuitVision','current_model':'Round 2 / YOLO11n / 57 classes',
 'training_performed':False,'datasets_requested':DATASETS,'datasets_audited':audits,'failures':failures,
 'important_warning':'These audit reports do NOT establish clean labels, no leakage or better detection accuracy. No model was trained.'}
json_path=ROOT/'CircuitVision_Dataset_Audit_Report.json'
json_path.write_text(json.dumps(report,indent=2,ensure_ascii=False))
csv_path=ROOT/'CircuitVision_Class_Counts.csv'
with csv_path.open('w',newline='') as fh:
    writer=csv.writer(fh);writer.writerow(['dataset','task','split','class','box_count'])
    for dataset in audits:
        for item in dataset.get('label_counts',[]):
            writer.writerow([dataset['name'],dataset['task'],item['split'],item['class'],item['boxes']])
archive=Path('/content/CircuitVision_Dataset_Audit_Report.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    z.write(json_path,json_path.name);z.write(csv_path,csv_path.name)
print('Created:',archive,'size:',archive.stat().st_size,'bytes')
from google.colab import files
files.download(str(archive))

## After the audit

Don't retrain yet. Have the class list and labels checked first. **Digital logic gate symbols are not the same as 74HC DIP IC photographs**. Keep the *physical-component detector* and *resistor-band color reader* separate so that their datasets don't corrupt each other's classes. Your existing CircuitVision website stays on verified Round 2 until a future model passes the frozen test set.